# AI Agente Analyze Fraud Rules 
Mich y cami
Ts
ops_fraud.falcon_declined_transactions fdt
Chargebacks.
Analytics_bi.transactions

User feedback: tabla, declined transactions.

Archivos PLT (v 2.0)

Diario -> dependemos de mich. 
Moverlo a transactions_ytd. 
Agregar 3ds a la tabla


transactions_


Order of Agents to implement
1. Detector de patrón: diario/semanal
- Suggestion: 
- Revise CB 

2. Agente de validación de reglas 
- Sheets - fraud rules


3. Revisión de CB, suggestion de nuevas falcon 
- reglas/limitantes de falcon
- Muy definido el output.
- Use this information de user feedback and suggest a change. 


In [1]:
cd ..

/Users/camila.cusicanqui/Documents/GitHub/frod-agentic-ai


In [7]:
from utils.data_ingest import get_db_conn
import pandas as pd
import numpy as np
import gspread
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import date, timedelta, datetime
pd.set_option('display.max_columns', None)
pd.options.display.float_format = '{:,.2f}'.format
import gspread


In [ ]:
fraud_rules_query = f"""
SELECT user_id, klrid, transaction_id, amount, timestamp_mx_created_at, prosa_timestamp, merchant, mcc_code, regla, pos_entry_mode, pin_capabilities, card_type, product_type
FROM ops_fraud.falcon_declined_transactions;
"""

In [5]:
fraud_rules_df = pd.read_sql_query(fraud_rules_query, get_db_conn())

/var/folders/ch/hb63cwpx22l_1jl5cqj7wp_80000gq/T/ipykernel_10298/1460050075.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  fraud_rules_df = pd.read_sql_query(fraud_rules_query, get_db_conn())


In [6]:
fraud_rules_df.head()

,user_id,klrid,transaction_id,amount,timestamp_mx_created_at,prosa_timestamp,merchant,mcc_code,regla,pos_entry_mode,pin_capabilities,card_type,product_type
0,4e9a92aa-cccf-4792-88f8-e73c46603e7e,01f41f54-8257-4bee-8bd2-e9bcd2a16590,PARABILIUM:124768024,-280.17,2025-08-05 00:03:16.361,2025-08-05 00:03:16.748,STRIPE AMAZON CIUDAD DE MEXCMXMX,5399,HF_High_Fraud_Score_General,CNP Manual,unknown,PHYSICAL,CREDIT_5456_BIN
1,4e9a92aa-cccf-4792-88f8-e73c46603e7e,01f41f54-8257-4bee-8bd2-e9bcd2a16590,PARABILIUM:124768030,-280.17,2025-08-05 00:03:17.153,2025-08-05 00:03:17.418,STRIPE AMAZON CIUDAD DE MEXCMXMX,5399,HF_High_Fraud_Score_General,CNP Manual,unknown,PHYSICAL,CREDIT_5456_BIN
2,52507396-3c60-4c60-987c-2547c1f06885,b5847975-7c66-4864-993e-8315acc9e40c,PARABILIUM:124768752,-250.00,2025-08-05 00:07:51.394,2025-08-05 00:07:51.656,NUVEIMXCALIENTE TIJUANA BCNMX,7941,FV___High_Fraud_Score_Online,CNP Card On File,unknown,VIRTUAL,CREDIT_5401_BIN
3,52507396-3c60-4c60-987c-2547c1f06885,b5847975-7c66-4864-993e-8315acc9e40c,PARABILIUM:124768869,-250.00,2025-08-05 00:08:32.801,2025-08-05 00:08:33.066,NUVEIMXCALIENTE TIJUANA BCNMX,7941,FV___High_Fraud_Score_Online,CNP Card On File,unknown,VIRTUAL,CREDIT_5401_BIN
4,52507396-3c60-4c60-987c-2547c1f06885,b5847975-7c66-4864-993e-8315acc9e40c,PARABILIUM:124768986,-250.00,2025-08-05 00:09:14.939,2025-08-05 00:09:15.196,NUVEIMXCALIENTE TIJUANA BCNMX,7941,FV___High_Fraud_Score_Online,CNP Card On File,unknown,VIRTUAL,CREDIT_5401_BIN


In [ ]:
SERVICE_ACCOUNT_FILE = r'.config/klar-cami-cusi.json'
gc = gspread.service_account(filename=SERVICE_ACCOUNT_FILE)
sheet_id = "1cQ8QzQqml1oBdYliYpW1V2ZquHJrH5iBZzwlGnYFV5I"
spreadsheet = gc.open_by_key(sheet_id)
# connect to ghseets
worksheet = spreadsheet.worksheet('All Falcon Rules')
falcon_rules_df = pd.DataFrame(worksheet.get_all_records())

# PROSA Rules prompt
TODO: 
- explain in prompt what each rule is and what it does,
- explain PROSA dynamics and how it works with the rules, and how the rules are applied in the transactions
- Each rule in the 'All Falcon Rules' worksheet represents a specific fraud prevention rule implement into Falcon, PROSA's fraud detection system. 
- These rules are designed to prevent fraudulent transactions by utilizing various factors and utilizing counter numeric variables.

Each rule has the following characteristics:
- Rule ID: an identifier to group similar rules.
- Rule name: identifier + general description of the rule.
- Logic: detailed rule definition.
- Rule type: the rules usually repeat certain general patterns. 
- Amount limit: if applies for rule, the limit of the amount implemented. 
- Score limit: each transaction in PROSA has a score. If this rule includes a threshold for the score, the limit of the amount implemented.
- Transaction limit: if applies to rule, the limit of how many transactions a user, or list can have. 
- Timeframe: timeeframe for which the rule applies. For example, if a merchant has a card validation merchant of GOOGLE and 24 hours later a TELCEL intent then the transaction does not pass. 
- Lists: There are lists that the rule can use. For example, in the case of MOContador it checks whether a card is inside a archivo de embozo.
- UDV: Each rule has a numeric rule, the numeric rules contain user defined variables that can store numbers (counter), dates, true or False. 
- Segment: The card portfolio group it belongs to.
- Subsegment: subsegment of the segment. 
- Motive: rule was created because of a recent fraud incident, or are general preventative rules.
- Last updated date: date of last update. 
- Creation date: date of rule creation.
- Estado: state of rule. 
- Comments: additional rule comments.

# Tools for pattern detector agent. 

We're going to prioritize the pattern detector agent. We run a weekly analysis of rules and it's motives for declining so, we need to check the following:
- Week by week what are the increases in declinations per rule
- Who are the merchants that are causing these casualites? 
- What's the behavior of the transaction like? Does the user have previous purchaes with this merchant? In velocity cases are they 
    - Is it multiples tries per user and then the trx passes?
    -
- Are there simlar behaviors between the declined transactions? 
- What are pos entry modes of the declined transactions? 
- What segment does the client belong to ?
- What are the stats of the merchant in the last year? last month ? last 7days? Do they significantly differ from the transactions that we declined?
- Does the merchant usually have 3ds, cvv present, not present? 
- Does the merchant currently have CB or active CB? 


# TODO:
- analyze mage analytics infra and see what class we can create for the tools agent manager 